In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import akshare as ak
from tqdm import tqdm
from datetime import datetime, timedelta
from collections import Counter
from quant_utils import filter_stock,filter_main_board,filter_main_board_no_st,add_ts_code
import tushare as ts
from tqdm.auto import tqdm
from pathlib import Path
from time import sleep
from datetime import date
import requests

BY_DATE_DIR = Path("data/by_date")
BY_STOCK_DIR = Path("data/by_stock")

BY_DATE_DIR.mkdir(parents=True, exist_ok=True)
BY_STOCK_DIR.mkdir(parents=True, exist_ok=True)

ts.set_token("103ef331836d4a8aa5d7485665c8f7503b8632c23748f1ba985c5713")
pro = ts.pro_api("103ef331836d4a8aa5d7485665c8f7503b8632c23748f1ba985c5713")

#YYMMDD format
today = date.today().strftime("%Y%m%d")


In [22]:
#抓取数据（每天凌晨12：00点自动化抓取）
stock_name = ak.stock_info_a_code_name()

stock_name

,code,name,board
0,000001,平安银行,深主板
1,000002,万 科Ａ,深主板
2,000004,*ST国华,深主板
3,000006,深振业Ａ,深主板
4,000007,全新好,深主板
...,...,...,...
5481,920978,开特股份,北交所
5482,920981,晶赛科技,北交所
5483,920982,锦波生物,北交所
5484,920985,海泰新能,北交所


In [23]:
stock_name

#加板块column
stock_name = filter_stock(stock_name)

#只要主板
main_board = filter_main_board(stock_name)

#加入ts_code
mainboard_with_ts_code = add_ts_code(main_board)

#过滤ST和*ST
no_ST_mainboard_with_ts_code = filter_main_board_no_st(mainboard_with_ts_code)

stock_name.to_csv("data/all_stock_name.csv", index=False, encoding="utf-8-sig")
main_board.to_csv("data/main_board.csv", index=False, encoding="utf-8-sig")
mainboard_with_ts_code.to_csv("data/main_board_with_ts_code.csv", index=False, encoding="utf-8-sig")
no_ST_mainboard_with_ts_code.to_csv("data/no_ST_mainboard_with_ts_code.csv", index=False, encoding="utf-8-sig")


In [24]:

def fetch_one_day_all_stocks_fast(stock_df, trade_date):
    """
    一次性抓取某一天全市场 daily + adj_factor，再和 stock_df 合并
    stock_df 需要包含: code, name, board, ts_code
    trade_date 格式: YYYYMMDD
    """
    daily_df = pro.daily(trade_date=trade_date)
    adj_df = pro.adj_factor(trade_date=trade_date)

    if daily_df is None or daily_df.empty:
        return pd.DataFrame()
    if adj_df is None or adj_df.empty:
        return pd.DataFrame()

    df = daily_df.merge(
        adj_df[["ts_code", "trade_date", "adj_factor"]],
        on=["ts_code", "trade_date"],
        how="left",
    )

    stock_info = stock_df.copy()
    stock_info["code"] = stock_info["code"].astype(str).str.zfill(6)

    df = df.merge(
        stock_info[["code", "name", "board", "ts_code"]],
        on="ts_code",
        how="inner",
    )

    df["trade_date"] = pd.to_datetime(df["trade_date"], format="%Y%m%d", errors="coerce")
    df = df.dropna(subset=["trade_date"])
    df = df.sort_values(["ts_code", "trade_date"]).reset_index(drop=True)
    return df


def update_one_day_store(one_day_df):
    if one_day_df is None or one_day_df.empty:
        print("当天没有数据可更新")
        return

    trade_date_str = one_day_df["trade_date"].iloc[0].strftime("%Y-%m-%d")

    # 按日期存
    date_path = BY_DATE_DIR / f"{trade_date_str}.parquet"
    one_day_df.to_parquet(date_path, index=False)

    # 按股票增量存
    for ts_code, sub_df in one_day_df.groupby("ts_code"):
        stock_path = BY_STOCK_DIR / f"{ts_code}.parquet"

        if stock_path.exists():
            old_df = pd.read_parquet(stock_path)
            merged = pd.concat([old_df, sub_df], ignore_index=True)
            merged = (
                merged.sort_values("trade_date")
                .drop_duplicates(subset=["trade_date"], keep="last")
                .reset_index(drop=True)
            )
        else:
            merged = (
                sub_df.sort_values("trade_date")
                .drop_duplicates(subset=["trade_date"], keep="last")
                .reset_index(drop=True)
            )

        merged.to_parquet(stock_path, index=False)


def run_daily_update_fast(stock_df, trade_date):
    one_day_df = fetch_one_day_all_stocks_fast(stock_df, trade_date)
    update_one_day_store(one_day_df)
    return one_day_df


In [25]:
today_df = run_daily_update_fast(mainboard_with_ts_code, trade_date=today)
print(today_df.shape)
display(today_df.head())


(3184, 15)


,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,adj_factor,code,name,board
0,000001.SZ,2026-03-03,10.85,10.95,10.80,10.88,10.85,0.03,0.2765,1028694.83,1119402.076,134.5794,000001,平安银行,深主板
1,000002.SZ,2026-03-03,4.75,4.78,4.66,4.67,4.75,-0.08,-1.6842,1596817.04,751480.501,181.7040,000002,万 科Ａ,深主板
2,000004.SZ,2026-03-03,6.23,6.23,6.23,6.23,6.56,-0.33,-5.0305,7328.00,4565.344,4.0640,000004,*ST国华,深主板
3,000006.SZ,2026-03-03,8.91,9.03,8.51,8.56,8.94,-0.38,-4.2506,298327.02,260234.437,39.7400,000006,深振业Ａ,深主板
4,000007.SZ,2026-03-03,13.66,13.73,12.58,12.60,13.50,-0.90,-6.6667,130824.51,169618.714,8.2840,000007,全新好,深主板


In [ ]:


payload = {
    "api_name": "stock_basic",
    "token": "你的token",
    "params": {},
    "fields": "ts_code,symbol,name"
}

try:
    r = requests.post("https://api.tushare.pro", json=payload, timeout=30)
    print("status:", r.status_code)
    print(r.text[:500])
except Exception as e:
    print("error:", repr(e))

status: 200
{"request_id":"","code":40101,"data":null,"msg":"您的token不对，请确认。"}


In [59]:
df = pd.read_parquet("/Users/ciciliu/stock_analysis/data/by_date/2000-01-10.parquet")
print(df.head())
df.to_csv("data/example.csv", index=False, encoding="utf-8-sig")

     ts_code trade_date   open   high    low  close  pre_close  change  \
0  000001.SZ 2000-01-10  19.79  20.48  19.77  20.14      19.54    0.60   
1  000002.SZ 2000-01-10  10.68  11.44  10.55  11.44      10.40    1.04   
2  000004.SZ 2000-01-10  10.28  10.36  10.01  10.36       9.87    0.49   
3  000006.SZ 2000-01-10  10.50  11.00  10.19  10.82      10.45    0.37   
4  000007.SZ 2000-01-10   9.50   9.99   9.45   9.61       9.33    0.28   

   pct_chg       vol       amount  adj_factor    code   name board  
0     3.07  185211.0  372294.4957      21.662  000001   平安银行   深主板  
1    10.00  142425.0  159259.0559       8.995  000002  万  科Ａ   深主板  
2     4.96   24623.0   25405.2053         NaN  000004  *ST国华   深主板  
3     3.54   27733.0   29816.0204       4.111  000006   深振业Ａ   深主板  
4     3.00   41236.0   39841.2720       3.632  000007    全新好   深主板  


In [13]:
main_codes = set(mainboard_with_ts_code["ts_code"].astype(str))
no_st_codes = set(no_ST_mainboard_with_ts_code["ts_code"].astype(str))

diff_codes = sorted(main_codes - no_st_codes)
print("difference count:", len(diff_codes))
print(diff_codes)

stock_meta = mainboard_with_ts_code.copy()
stock_meta["ts_code"] = stock_meta["ts_code"].astype(str)

st_meta = stock_meta[stock_meta["ts_code"].isin(diff_codes)].copy()
st_meta["code"] = st_meta["code"].astype(str).str.zfill(6)

print("symbols to backfill:", len(st_meta))
print(st_meta[["ts_code", "name"]].head())

difference count: 129
['000004.SZ', '000430.SZ', '000488.SZ', '000504.SZ', '000518.SZ', '000595.SZ', '000608.SZ', '000609.SZ', '000615.SZ', '000638.SZ', '000656.SZ', '000668.SZ', '000669.SZ', '000691.SZ', '000697.SZ', '000698.SZ', '000711.SZ', '000736.SZ', '000752.SZ', '000793.SZ', '000820.SZ', '000821.SZ', '000903.SZ', '000908.SZ', '000909.SZ', '000929.SZ', '000972.SZ', '001270.SZ', '002005.SZ', '002024.SZ', '002047.SZ', '002055.SZ', '002058.SZ', '002076.SZ', '002122.SZ', '002168.SZ', '002199.SZ', '002200.SZ', '002211.SZ', '002214.SZ', '002231.SZ', '002253.SZ', '002289.SZ', '002305.SZ', '002306.SZ', '002424.SZ', '002485.SZ', '002496.SZ', '002512.SZ', '002528.SZ', '002529.SZ', '002569.SZ', '002581.SZ', '002586.SZ', '002592.SZ', '002620.SZ', '002630.SZ', '002647.SZ', '002650.SZ', '002656.SZ', '002689.SZ', '002693.SZ', '002713.SZ', '002717.SZ', '002731.SZ', '002742.SZ', '002762.SZ', '002789.SZ', '002808.SZ', '002816.SZ', '002822.SZ', '002848.SZ', '002868.SZ', '002872.SZ', '002898.SZ', '0

In [15]:
def fetch_one_stock_full_history(ts_code, stock_meta_df):
    daily_df = pro.daily(ts_code=ts_code)
    adj_df = pro.adj_factor(ts_code=ts_code)

    if daily_df is None or daily_df.empty:
        return pd.DataFrame()
    if adj_df is None or adj_df.empty:
        return pd.DataFrame()

    df = daily_df.merge(
        adj_df[["ts_code", "trade_date", "adj_factor"]],
        on=["ts_code", "trade_date"],
        how="left",
    )

    meta = stock_meta_df[stock_meta_df["ts_code"] == ts_code][["code", "name", "board", "ts_code"]].copy()

    df = df.merge(meta, on="ts_code", how="inner")

    df["trade_date"] = pd.to_datetime(df["trade_date"], format="%Y%m%d", errors="coerce")
    df = df.dropna(subset=["trade_date"])
    df = df.sort_values(["trade_date"]).reset_index(drop=True)
    return df


def update_stock_and_dates(stock_df):
    if stock_df is None or stock_df.empty:
        return

    ts_code = stock_df["ts_code"].iloc[0]

    # Update by_stock
    stock_path = BY_STOCK_DIR / f"{ts_code}.parquet"
    if stock_path.exists():
        old_df = pd.read_parquet(stock_path)
        merged_stock = pd.concat([old_df, stock_df], ignore_index=True)
        merged_stock = (
            merged_stock
            .sort_values("trade_date")
            .drop_duplicates(subset=["trade_date"], keep="last")
            .reset_index(drop=True)
        )
    else:
        merged_stock = (
            stock_df
            .sort_values("trade_date")
            .drop_duplicates(subset=["trade_date"], keep="last")
            .reset_index(drop=True)
        )

    merged_stock.to_parquet(stock_path, index=False)

    # Update by_date
    for trade_date, sub_df in stock_df.groupby("trade_date"):
        date_str = trade_date.strftime("%Y-%m-%d")
        date_path = BY_DATE_DIR / f"{date_str}.parquet"

        if date_path.exists():
            old_date_df = pd.read_parquet(date_path)
            merged_date = pd.concat([old_date_df, sub_df], ignore_index=True)
            merged_date = (
                merged_date
                .sort_values(["ts_code", "trade_date"])
                .drop_duplicates(subset=["ts_code", "trade_date"], keep="last")
                .reset_index(drop=True)
            )
        else:
            merged_date = (
                sub_df
                .sort_values(["ts_code", "trade_date"])
                .drop_duplicates(subset=["ts_code", "trade_date"], keep="last")
                .reset_index(drop=True)
            )

        merged_date.to_parquet(date_path, index=False)

In [19]:

failed_symbols = []

for ts_code in tqdm(diff_codes, desc="Backfilling ST stocks"):
    try:
        stock_df = fetch_one_stock_full_history(ts_code, st_meta)
        if stock_df.empty:
            print(f"{ts_code}: no data")
        else:
            update_stock_and_dates(stock_df)
        sleep(1.5)
    except Exception as e:
        failed_symbols.append({"ts_code": ts_code, "error": str(e)})
        print(f"{ts_code}: failed - {e}")
        sleep(5)

print("failed symbols:", len(failed_symbols))
failed_symbols

Backfilling ST stocks:   0%|          | 0/129 [00:00<?, ?it/s]

failed symbols: 0


[]

In [51]:

daily_000001 = pro.daily(ts_code="000001.SZ",start_date="19910404",end_date = "19910909").sort_values("trade_date").reset_index(drop=True)

daily_000001

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,000001.SZ,19910404,48.76,48.76,48.76,48.76,49.00,-0.24,-0.49,3.0,15.0
1,000001.SZ,19910405,48.52,48.52,48.52,48.52,48.76,-0.24,-0.49,2.0,10.0
2,000001.SZ,19910408,48.04,48.04,48.04,48.04,48.52,-0.48,-0.99,2.0,10.0
3,000001.SZ,19910409,47.80,47.80,47.80,47.80,48.04,-0.24,-0.50,4.0,19.0
4,000001.SZ,19910410,47.56,47.56,47.56,47.56,47.80,-0.24,-0.50,15.0,71.0
...,...,...,...,...,...,...,...,...,...,...,...
83,000001.SZ,19910903,15.00,15.00,14.70,15.00,15.00,0.00,0.00,2874.0,4271.0
84,000001.SZ,19910904,14.80,14.85,14.70,14.85,15.00,-0.15,-1.00,1604.0,2367.0
85,000001.SZ,19910905,14.70,14.70,14.30,14.60,14.85,-0.25,-1.68,4840.0,7030.0
86,000001.SZ,19910906,14.30,14.30,13.50,13.70,14.60,-0.90,-6.16,3267.0,4506.0


In [39]:
list_date_000004 = pro.stock_basic(ts_code="000004.SZ", fields="ts_code,list_date")
print(list_date_000004)

     ts_code list_date
0  000004.SZ  19901201
